# DeepSeek OCR API Server

Deploy DeepSeek VL2 model as an OCR API on Google Colab.
The local `pdf-extractor.py` sends page images to this endpoint.

**Usage:**
1. Run all cells in order
2. Copy the ngrok URL printed in the last cell
3. Set `DEEPSEEK_OCR_URL=<ngrok_url>/ocr` in your `.env` file

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers torch flask pyngrok Pillow

In [ ]:
# Cell 2: Load DeepSeek VL2 model for OCR
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/deepseek-vl2-tiny"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).cuda()
model.eval()
print("Model loaded successfully!")

In [ ]:
# Cell 3: Define OCR inference function
import base64
import io
from PIL import Image


def run_ocr(image_b64: str) -> str:
    """Run OCR on a base64-encoded image using DeepSeek VL2."""
    image_bytes = base64.b64decode(image_b64)
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # Prompt the model to extract Vietnamese text
    prompt = "Extract all text from this image. Return the text exactly as it appears, preserving the original language (Vietnamese)."

    # Model-specific inference (adjust based on DeepSeek VL2 API)
    inputs = tokenizer(
        prompt,
        images=[image],
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
        )

    # Decode only the new tokens (skip the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()

In [ ]:
# Cell 4: Flask API server + ngrok tunnel
import threading
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)


@app.route("/ocr", methods=["POST"])
def ocr_endpoint():
    """Receive base64 image, return extracted text."""
    data = request.json
    if not data or "image" not in data:
        return jsonify({"error": "Missing 'image' field"}), 400

    try:
        text = run_ocr(data["image"])
        return jsonify({"text": text})
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": MODEL_NAME})


# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\n{'='*60}")
print(f"OCR API URL: {public_url}/ocr")
print(f"Health check: {public_url}/health")
print(f"{'='*60}")
print(f"\nSet in .env: DEEPSEEK_OCR_URL={public_url}/ocr")

# Run Flask in a thread so the cell doesn't block
threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False),
    daemon=True,
).start()
print("\nServer running! Keep this notebook open.")